## 1. Setup and Imports

In [1]:
# Cell 1 - Installing Libraries and Importing Packages
%pip install rembg[gpu] gradio tensorflow tensorflow_hub opencv-python --quiet

import gradio as gr
from PIL import Image
import numpy as np
from rembg import remove, new_session
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import math

print("Libraries installed and imported successfully.")

Note: you may need to restart the kernel to use updated packages.


/mnt/c/Users/pasca/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-06 21:26:33.424713370 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
2025-10-06 21:27:10.429220: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 21:27:20.905472: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instru

Libraries installed and imported successfully.


## 2. AI Model Loading and Setup

In [2]:
# Cell 2 - AI Model Loading and Setup
print("Loading Super-Resolution model from TensorFlow Hub...")
SUPER_RESOLUTION_MODEL_URL = "https://tfhub.dev/captain-pool/esrgan-tf2/1"
super_res_model = hub.load(SUPER_RESOLUTION_MODEL_URL)
print("Super-Resolution model loaded.")

print("Preparing Anime Background Removal model session...")
anime_session = new_session("isnet-anime")
print("All models ready.")

Loading Super-Resolution model from TensorFlow Hub...


I0000 00:00:1759760934.749868   25139 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5518 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Super-Resolution model loaded.
Preparing Anime Background Removal model session...
All models ready.


2025-10-06 21:28:57.291065566 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcurand.so.10: cannot open shared object file: No such file or directory



## 3. AI Feature Definitions

In [3]:
# Cell 3 - AI Feature Definitions
# --- Feature 1: Background Removal (Now with model selection) ---
def remove_background(input_image: Image.Image, model_choice: str) -> Image.Image:
    if input_image is None: return None
    if model_choice == "Anime (isnet-anime)":
        return remove(input_image, session=anime_session)
    else:
        return remove(input_image)

# --- Feature 2: Super-Resolution (Now with multi-pass and tiling) ---
def enhance_resolution_tiled(input_image: Image.Image, model, target_scale: int, tile_size: int = 256, overlap: int = 32) -> Image.Image:
    if input_image is None: return None
    MODEL_BASE_SCALE = 4
    num_passes = int(math.log(target_scale, MODEL_BASE_SCALE))
    current_image = input_image
    for i in range(num_passes):
        print(f"Starting upscale pass {i+1}/{num_passes}...")
        width, height = current_image.size
        output_image = Image.new('RGB', (width * MODEL_BASE_SCALE, height * MODEL_BASE_SCALE))
        step = tile_size - overlap
        for y in range(0, height, step):
            for x in range(0, width, step):
                bbox = (x, y, min(x + tile_size, width), min(y + tile_size, height))
                tile = current_image.crop(bbox)
                img_np = cv2.cvtColor(np.array(tile), cv2.COLOR_RGB2BGR)
                img_tf = tf.expand_dims(tf.cast(img_np, tf.float32), 0)
                upscaled_tensor = model(img_tf)
                upscaled_tensor = tf.clip_by_value(upscaled_tensor, 0, 255)
                upscaled_image_np = tf.cast(tf.squeeze(upscaled_tensor), tf.uint8).numpy()
                upscaled_tile = Image.fromarray(cv2.cvtColor(upscaled_image_np, cv2.COLOR_BGR2RGB))
                output_image.paste(upscaled_tile, (x * MODEL_BASE_SCALE, y * MODEL_BASE_SCALE))
        current_image = output_image
        print(f"Upscale pass {i+1} complete. Image is now {current_image.width}x{current_image.height}.")
    return current_image

## 4. Main Processing Function and Gradio UI

In [4]:
# Cell 4 - Main Processing Function and Gradio UI (Fully Updated)

def process_image(input_image: Image.Image, tool_choice: str, bg_model: str, sr_scale: int) -> Image.Image:
    """Applies the selected AI tool with the specified options."""
    if input_image is None:
        return None

    if tool_choice == "Background Removal":
        return remove_background(input_image, bg_model)
    elif tool_choice == "Super-Resolution (Enhance)":
        return enhance_resolution_tiled(input_image, super_res_model, target_scale=sr_scale)
    else:
        return input_image

# --- NEW: Function to control UI visibility ---
def update_tool_options(tool_choice):
    """Shows/hides advanced options based on the selected tool."""
    if tool_choice == "Background Removal":
        # Show the background model dropdown, hide the super-res radio
        return gr.update(visible=True), gr.update(visible=False)
    elif tool_choice == "Super-Resolution (Enhance)":
        # Show the super-res radio, hide the background model dropdown
        return gr.update(visible=False), gr.update(visible=True)
    else:
        # Hide both if no tool is selected
        return gr.update(visible=False), gr.update(visible=False)

# --- NEW: Updated Gradio UI with Dynamic Controls ---
with gr.Blocks(theme=gr.themes.Soft()) as iface:
    gr.Markdown("# 🌟 Smart Image Editing Application (Mini-Adobe AI)")
    gr.Markdown("Upload an image and select an AI tool to process it. Advanced options will appear based on your tool selection.")
    
    with gr.Row():
        with gr.Column(scale=2):
            input_img = gr.Image(type="pil", label="Upload Your Image")
            tool_choice = gr.Dropdown(
                ["Background Removal", "Super-Resolution (Enhance)"], 
                label="Choose AI Tool"
            )

            # --- Advanced Options Container ---
            with gr.Column() as advanced_options:
                # This option is for Background Removal, hidden by default
                bg_model_dd = gr.Dropdown(
                    ["General (u2net)", "Anime (isnet-anime)"], 
                    label="Background Removal Model", 
                    value="General (u2net)",
                    visible=False # Starts hidden
                )
                # This option is for Super-Resolution, hidden by default and now a Radio
                sr_scale_radio = gr.Radio(
                    [4, 16],
                    label="Super-Resolution Scale Factor",
                    value=4,
                    info="Select the desired upscaling factor (e.g., 4x or 16x).",
                    visible=False # Starts hidden
                )
            
            submit_btn = gr.Button("Submit", variant="primary")
            
        with gr.Column(scale=3):
            output_img = gr.Image(type="pil", label="Processed Image")

    # --- NEW: Event listener to make the UI dynamic ---
    tool_choice.change(
        fn=update_tool_options,
        inputs=tool_choice,
        outputs=[bg_model_dd, sr_scale_radio] # The components to update
    )

    submit_btn.click(
        fn=process_image,
        inputs=[input_img, tool_choice, bg_model_dd, sr_scale_radio],
        outputs=output_img
    )

# Launch the interface
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://51560519168a942e66.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2025-10-06 21:29:25.321333346 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcurand.so.10: cannot open shared object file: No such file or directory



Starting upscale pass 1/1...


2025-10-06 21:36:13.480792: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


Upscale pass 1 complete. Image is now 2472x2472.
